In [26]:
# Imports section
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [27]:
# Using pandas load the dataset (load remotely, not locally)
data = pd.read_csv('science_data_large.csv')
# Output the first 15 rows of the data
print("First 15 rows")
print(data.head(15))
# Display a summary of the table information (number of datapoints, etc.)
print('\nSummary')
print(data.describe())

First 15 rows
    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04

Summary
       Temperature °C     Mols KCL     Size nm^3
count     1000.000000  1000.000000  1.000000e+03
mean       500.500000   471.530000  5.086111e+05
std        288.819436   288.482872  4.474838e+05
min          1.000000     1.000000  1.611429e+01
25%        250.750000   226.750000  1.298267

## Part 2. Splitting the dataset

In [28]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = data.drop(columns=['Size nm^3'])
y = data['Size nm^3']
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

## Part 3. Perform a Linear Regression

In [29]:
# Use sklearn to train a model on the training set
model = LinearRegression()
model.fit(X_train, y_train)
# Create a sample datapoint and predict the output of that sample with the trained model
sample_datapoint = X_test.iloc[0:1]
print('Sample:', sample_datapoint)
predicted_output = model.predict(sample_datapoint)
print('Prediction:', predicted_output)
# Report on the score for that model, in your own words (markdown, not code) explain what the score means
score = model.score(X_test, y_test)
print('Score:', score)
# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = model.coef_
intercept = model.intercept_
print('Intercept:', intercept, '\nCoefficients:', coefficients)

Sample:      Temperature °C  Mols KCL
521             100       541
Prediction: [235911.1927226]
Score: 0.8552472077276095
Intercept: -409391.47958340764 
Coefficients: [ 866.14641337 1032.69506649]


For the selected sample datapoint of Temperature = 100 and Mols = 541, the model predicts 235910

The score of the prediction was 0.85525, since it is close to 1, the model is doing a good job of explaining the variability of the target variable.

$h(x) = -409390 + 866.15x_1 + 1032.70x_2$

## Part 4. Use Cross Validation

In [30]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
scores = cross_val_score(model, X, y, cv=10)
# Report on their finding and their significance
mean_score = scores.mean()
std_dev = scores.std()

print(f"Cross-validation scores: {scores}")
print(f"Mean score (R²): {mean_score}")
print(f"Standard deviation of scores: {std_dev}")

Cross-validation scores: [0.81123596 0.86440978 0.87808742 0.86561069 0.87495621 0.84484397
 0.87941022 0.86349411 0.78353682 0.88686516]
Mean score (R²): 0.8552450341984701
Standard deviation of scores: 0.03152876296534241


The mean score was 0.85525 and the standard deviation was 0.03153. This means that on average the model was a good fit and because the standard deviation was low, it is consistent across different data splits.

## Part 5. Using Polynomial Regression

In [37]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)
# Report on the metrics and output the resultant equation as you did in Part 3.
poly_sample_datapoint = X_test_poly[0:1]
poly_prediction = model_poly.predict(poly_sample_datapoint)
print('Prediction:', poly_prediction)

poly_score = model_poly.score(X_test_poly, y_test)
print('Score:', poly_score)

# Extract the intercept and coefficients
poly_intercept = model_poly.intercept_
poly_coefficients = model_poly.coef_

print(poly_intercept)
print(poly_coefficients)
# Get feature names for the polynomial terms
feature_names = poly.get_feature_names_out(input_features=X.columns)

# Constructing the polynomial equation
equation_parts = [f"{poly_intercept:.3f}"]  # Start with the intercept

# Add each coefficient and corresponding feature to the equation
for coef, name in zip(poly_coefficients, feature_names):
    if coef != 0:  # Only add terms with a non-zero coefficient
        equation_parts.append(f"{coef:.3f} * {name}")

# Join all parts into a single string to represent the equation
polynomial_equation = " + ".join(equation_parts)

print(f"Polynomial regression equation: h(x) = {polynomial_equation}")

Prediction: [117762.31427637]
Score: 1.0
2.0477105863392353e-05
[ 0.00000000e+00  1.20000000e+01 -1.27195488e-07  1.26494371e-11
  2.00000000e+00  2.85714287e-02]
Polynomial regression equation: h(x) = 0.000 + 12.000 * Temperature °C + -0.000 * Mols KCL + 0.000 * Temperature °C^2 + 2.000 * Temperature °C Mols KCL + 0.029 * Mols KCL^2


For the selected sample datapoint of Temperature = 100 and Mols = 541, the model predicts 117760

The score was a perfect 1.0, meaning that the model perfectly explains all variation in the dependent variable.

$h(x) = 12.0 \cdot x_1 + 2x_1x_2+0.029x_2^2$